In [7]:
import pandas as pd
import requests
from datetime import datetime, timedelta

def download_smard_prices(start_date, end_date):
    """
    Download day-ahead prices from SMARD API
    
    Parameters:
    - start_date: 'YYYY-MM-DD' format
    - end_date: 'YYYY-MM-DD' format
    """
    
    all_data = []
    
    # SMARD API returns data in weekly chunks, need to iterate
    current_date = datetime.strptime(start_date, '%Y-%m-%d')
    end_datetime = datetime.strptime(end_date, '%Y-%m-%d')
    
    while current_date < end_datetime:
        # Convert to timestamp (milliseconds)
        timestamp = int(current_date.timestamp() * 1000)
        
        # Correct URL - Day-ahead prices Germany/Luxembourg (filter 4169)
        url = "https://www.smard.de/app/chart_data/4169/DE/index_hour.json"
        
        print(f"Fetching data for week of {current_date.strftime('%Y-%m-%d')}...")
        
        try:
            response = requests.get(url, timeout=30)
            
            if response.status_code == 200:
                data = response.json()
                series = data.get('series', [])
                
                for item in series:
                    if item[1] is not None:  # Skip null values
                        all_data.append({
                            'timestamp': item[0],
                            'price': item[1]
                        })
            else:
                print(f"  Warning: Status {response.status_code}")
                
        except Exception as e:
            print(f"  Error: {e}")
        
        # Move to next week
        current_date += timedelta(days=7)
    
    if not all_data:
        print("No data retrieved")
        return None
    
    # Convert to DataFrame
    df = pd.DataFrame(all_data)
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    
    # Remove duplicates and sort
    df = df.drop_duplicates(subset='timestamp')
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    # Filter to requested date range
    df = df[(df['timestamp'] >= start_date) & (df['timestamp'] <= end_date)]
    
    return df


# Download 6 months of data
print("Starting download from SMARD API...")
print("=" * 50)

df = download_smard_prices('2025-06-01', '2025-12-31')

if df is not None:
    print("=" * 50)
    print(f"✓ Downloaded {len(df)} rows")
    print(f"✓ Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    print(f"✓ Price range: €{df['price'].min():.2f} to €{df['price'].max():.2f}")
    
    # Save to CSV
    df.to_csv('data/electricity_prices.csv', index=False)
    print(f"✓ Saved to data/electricity_prices.csv")


Starting download from SMARD API...
Fetching data for week of 2025-06-01...
Fetching data for week of 2025-06-08...
Fetching data for week of 2025-06-15...
Fetching data for week of 2025-06-22...
Fetching data for week of 2025-06-29...
Fetching data for week of 2025-07-06...
Fetching data for week of 2025-07-13...
Fetching data for week of 2025-07-20...
Fetching data for week of 2025-07-27...
Fetching data for week of 2025-08-03...
Fetching data for week of 2025-08-10...
Fetching data for week of 2025-08-17...
Fetching data for week of 2025-08-24...
Fetching data for week of 2025-08-31...
Fetching data for week of 2025-09-07...
Fetching data for week of 2025-09-14...
Fetching data for week of 2025-09-21...
Fetching data for week of 2025-09-28...
Fetching data for week of 2025-10-05...
Fetching data for week of 2025-10-12...
Fetching data for week of 2025-10-19...
Fetching data for week of 2025-10-26...
Fetching data for week of 2025-11-02...
Fetching data for week of 2025-11-09...
Fetc

In [4]:
import pandas as pd
import requests

def download_energy_charts_prices(year):
    """
    Download day-ahead prices from Energy-Charts (Fraunhofer ISE)
    """
    
    url = f"[api.energy-charts.info](https://api.energy-charts.info/price?bzn=DE-LU&year={year})"
    
    print(f"Fetching {year} data from Energy-Charts...")
    
    response = requests.get(url, timeout=60)
    
    if response.status_code == 200:
        data = response.json()
        
        # Extract timestamps and prices
        timestamps = data.get('unix_seconds', [])
        prices = data.get('price', [])
        
        df = pd.DataFrame({
            'timestamp': pd.to_datetime(timestamps, unit='s'),
            'price': prices
        })
        
        # Remove any null prices
        df = df.dropna(subset=['price'])
        
        return df
    else:
        print(f"Error: {response.status_code}")
        return None


# Download 2024 data
df = download_energy_charts_prices(2024)

if df is not None:
    print(f"✓ Downloaded {len(df)} rows")
    print(f"✓ Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    print(f"✓ Price range: €{df['price'].min():.2f} to €{df['price'].max():.2f}")
    
    df.to_csv('data/electricity_prices.csv', index=False)
    print(f"✓ Saved to data/electricity_prices.csv")


Fetching 2024 data from Energy-Charts...


InvalidSchema: No connection adapters were found for '[api.energy-charts.info](https://api.energy-charts.info/price?bzn=DE-LU&year=2024)'

In [5]:
import pandas as pd

# Energy-Charts allows direct CSV export via URL parameters
url = "[energy-charts.info](https://energy-charts.info/charts/price_spot_market/data/de/year_2024.json)"

response = requests.get(url)
if response.status_code == 200:
    print("Data retrieved successfully")
    # Parse based on response structure


InvalidSchema: No connection adapters were found for '[energy-charts.info](https://energy-charts.info/charts/price_spot_market/data/de/year_2024.json)'

In [8]:
import pandas as pd
import requests
from datetime import datetime

def download_smard_prices(start_date, end_date):
    all_data = []

    print("Fetching available timestamps...")

    # Step 1: Get valid timestamps
    index_url = "https://www.smard.de/app/chart_data/4169/DE/index_hour.json"
    response = requests.get(index_url, timeout=30)

    if response.status_code != 200:
        print("Failed to fetch index")
        return None

    data = response.json()
    timestamps = data.get("timestamps", [])

    if not timestamps:
        print("No timestamps found")
        return None

    print(f"Found {len(timestamps)} timestamps")

    # Convert user dates
    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)

    # Step 2: Loop through valid timestamps only
    for ts in timestamps:
        dt = pd.to_datetime(ts, unit='ms')

        if dt < start_dt or dt > end_dt:
            continue

        url = f"https://www.smard.de/app/chart_data/4169/DE/4169_DE_hour_{ts}.json"

        print(f"Fetching {dt.date()}...")

        try:
            r = requests.get(url, timeout=30)

            if r.status_code != 200:
                continue

            json_data = r.json()
            series = json_data.get("series", [])

            for item in series:
                if item[1] is not None:
                    all_data.append({
                        "timestamp": item[0],
                        "price": item[1]
                    })

        except Exception as e:
            print(f"Error: {e}")

    if not all_data:
        print("No data retrieved")
        return None

    # Step 3: DataFrame
    df = pd.DataFrame(all_data)
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

    df = df.drop_duplicates(subset="timestamp")
    df = df.sort_values("timestamp").reset_index(drop=True)

    return df


# RUN
df = download_smard_prices("2025-06-01", "2025-12-31")

if df is not None:
    print(f"\nDownloaded {len(df)} rows")
    print(df.head())

    df.to_csv("electricity_prices.csv", index=False)

Fetching available timestamps...
Found 392 timestamps
Fetching 2025-06-01...
Fetching 2025-06-08...
Fetching 2025-06-15...
Fetching 2025-06-22...
Fetching 2025-06-29...
Fetching 2025-07-06...
Fetching 2025-07-13...
Fetching 2025-07-20...
Fetching 2025-07-27...
Fetching 2025-08-03...
Fetching 2025-08-10...
Fetching 2025-08-17...
Fetching 2025-08-24...
Fetching 2025-08-31...
Fetching 2025-09-07...
Fetching 2025-09-14...
Fetching 2025-09-21...
Fetching 2025-09-28...
Fetching 2025-10-05...
Fetching 2025-10-12...
Fetching 2025-10-19...
Fetching 2025-10-26...
Fetching 2025-11-02...
Fetching 2025-11-09...
Fetching 2025-11-16...
Fetching 2025-11-23...
Fetching 2025-11-30...
Fetching 2025-12-07...
Fetching 2025-12-14...
Fetching 2025-12-21...
Fetching 2025-12-28...

Downloaded 5209 rows
            timestamp  price
0 2025-06-01 22:00:00  81.76
1 2025-06-01 23:00:00  75.31
2 2025-06-02 00:00:00  75.00
3 2025-06-02 01:00:00  77.75
4 2025-06-02 02:00:00  83.05
